# 📉 Notebook 04 — Statistical Analysis

**Project:** Nigeria Disease Surveillance Dashboard  
**Purpose:** Apply formal statistical tests to the surveillance data.

---
**Tests applied:**
1. Mann-Kendall trend test — is incidence rising or falling?
2. Kruskal-Wallis seasonality test — is there a seasonal pattern?
3. Spearman correlation — rainfall vs. disease burden
4. CUSUM outbreak detection — flag unusual weekly spikes
5. K-means state clustering — group states by burden profile
6. CFR benchmarking — states with significantly higher mortality


## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Paths, Diseases
from src.analysis.statistics import (
    add_rolling_metrics,
    test_trend,
    test_seasonality,
    test_correlation,
    detect_outbreaks,
    outbreak_alerts_to_dataframe,
    cluster_states,
    benchmark_cfr,
)

# Load master data
df = pd.read_csv(
    Paths.processed / 'master_surveillance.csv',
    parse_dates=['report_date']
)
df['year']   = df['report_date'].dt.year
df['month']  = df['report_date'].dt.month
df['season'] = df['month'].apply(
    lambda m: 'Dry' if m in [11,12,1,2,3] else 'Rainy'
)

# Load rainfall for correlation analysis
rain_path = Paths.processed / 'rainfall_clean.csv'
rain_df   = pd.read_csv(rain_path) if rain_path.exists() else pd.DataFrame()

print(f'✅ Loaded: {len(df):,} rows | {df["disease"].nunique()} diseases')
print(f'   Rainfall available: {not rain_df.empty}')

## 2. Rolling Metrics

Add 4-week rolling average and week-on-week percentage change to the master DataFrame.

In [ ]:
df = add_rolling_metrics(df)

print('✅ Rolling metrics added')
print(f'   cases_4wk_avg   : {df["cases_4wk_avg"].notna().sum():,} non-null values')
print(f'   pct_change_wow  : {df["pct_change_wow"].notna().sum():,} non-null values')

# Visualise rolling average vs raw for cholera
cholera_df = (
    df[df['disease'] == Diseases.CHOLERA]
    .groupby('report_date')
    .agg(confirmed_cases=('confirmed_cases','sum'),
         cases_4wk_avg=('cases_4wk_avg','mean'))
    .reset_index()
    .sort_values('report_date')
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cholera_df['report_date'], y=cholera_df['confirmed_cases'],
    name='Raw weekly cases', line=dict(color='#cccccc', width=1)
))
fig.add_trace(go.Scatter(
    x=cholera_df['report_date'], y=cholera_df['cases_4wk_avg'],
    name='4-week rolling avg', line=dict(color='#1D9E75', width=2)
))
fig.update_layout(
    title='Cholera — Raw Cases vs 4-Week Rolling Average',
    xaxis_title='Date', yaxis_title='Confirmed Cases',
    template='plotly_white', height=350
)
fig.show()

## 3. Mann-Kendall Trend Test

Non-parametric test for monotonic trends. Does not assume normality.

In [ ]:
print('Running Mann-Kendall trend tests...\n')

trend_results = []
for disease in df['disease'].unique():
    result = test_trend(df, disease=disease, state=None)
    trend_results.append(result)

    sig_marker = '⭐' if result.significant else '  '
    print(f'  {sig_marker} {disease:<20} {result.trend:<15} '
          f'τ={result.tau:+.3f}  p={result.p_value:.4f}')

print()
print('  ⭐ = statistically significant at α=0.05')
print()
for r in trend_results:
    if r.significant:
        print(f'  → {r.interpretation}')

## 4. Kruskal-Wallis Seasonality Test

In [ ]:
print('Testing for seasonal patterns...\n')

for disease in df['disease'].unique():
    result = test_seasonality(df, disease=disease, state=None)
    sig_marker = '⭐' if result.significant else '  '
    print(f'  {sig_marker} {disease:<20} '
          f'H={result.h_statistic:.2f}  p={result.p_value:.4f}  '
          f'peak={result.peak_season}')

# Plot dry vs rainy season comparison for cholera
seasonal_data = (
    df[df['disease'] == Diseases.CHOLERA]
    .groupby('season')['confirmed_cases']
    .agg(['mean','median','std'])
    .reset_index()
)

fig = px.bar(
    seasonal_data,
    x='season', y='mean',
    color='season',
    error_y='std',
    color_discrete_map={'Dry':'#EF9F27','Rainy':'#185FA5'},
    title='Cholera — Mean Weekly Cases: Dry vs Rainy Season',
    labels={'mean':'Avg Weekly Cases','season':'Season'},
    template='plotly_white',
)
fig.update_layout(height=320, showlegend=False)
fig.show()

## 5. Spearman Correlation — Rainfall vs. Disease

In [ ]:
if rain_df.empty:
    print('⚠️  Rainfall data not available — skipping correlation test')
    print('   Run the NASA rainfall extraction in Notebook 01')
else:
    # Merge rainfall with monthly disease aggregates
    monthly = (
        df.dropna(subset=['year','month'])
        .groupby(['state','disease','year','month'])['confirmed_cases']
        .sum()
        .reset_index()
    )
    merged_rain = monthly.merge(
        rain_df[['state','year','month','rainfall_mm']],
        on=['state','year','month'], how='left'
    )

    print('Spearman correlation: rainfall vs. disease burden\n')
    for disease in [Diseases.CHOLERA, Diseases.MENINGITIS]:
        result = test_correlation(
            merged_rain,
            disease          = disease,
            covariate_col    = 'rainfall_mm',
            covariate_label  = 'monthly rainfall',
        )
        sig_marker = '⭐' if result.significant else '  '
        direction_icon = '↑' if result.direction=='positive' else '↓' if result.direction=='negative' else '—'
        print(f'  {sig_marker} {disease:<20} ρ={result.rho:+.3f}  p={result.p_value:.4f}  {direction_icon}')
        if result.significant:
            print(f'     {result.interpretation}')

## 6. CUSUM Outbreak Detection

In [ ]:
print('Running CUSUM outbreak detection...\n')

all_alerts = []
for disease in df['disease'].unique():
    alerts = detect_outbreaks(
        df, disease=disease,
        threshold_multiplier=2.0,
        baseline_weeks=52,
    )
    all_alerts.extend(alerts)
    print(f'  {disease:<20} {len(alerts):>3} outbreak alerts')

alerts_df = outbreak_alerts_to_dataframe(all_alerts)
print(f'\n  Total alerts: {len(alerts_df)}')

if not alerts_df.empty:
    print('\n  Top 10 most severe alerts (by CUSUM score):')
    display(
        alerts_df.nlargest(10, 'cusum_score')[[
            'state','disease','alert_date','cases','baseline_mean','cusum_score'
        ]]
    )

## 7. K-Means State Clustering

In [ ]:
print('Running K-means clustering (k=4)...\n')

cluster_result = cluster_states(
    df, disease=Diseases.CHOLERA, n_clusters=4
)

if not cluster_result.state_clusters.empty:
    fig = px.scatter(
        cluster_result.state_clusters,
        x     = 'avg_incidence',
        y     = 'avg_cfr',
        color = 'cluster_label',
        size  = 'total_cases',
        text  = 'state',
        title = f'State Clustering — Cholera Burden Profile (k=4)',
        labels = {'avg_incidence':'Avg Incidence /100k','avg_cfr':'Avg CFR (%)'},
        color_discrete_sequence = ['#1D9E75','#185FA5','#EF9F27','#993C1D'],
        template = 'plotly_white',
    )
    fig.update_traces(textposition='top center', textfont_size=9)
    fig.update_layout(height=480)
    fig.show()

    print('\n  Cluster profiles:')
    display(cluster_result.cluster_profiles)

## 8. CFR Benchmarking

In [ ]:
print('CFR benchmarking by disease...\n')

for disease in df['disease'].unique():
    cfr_df = benchmark_cfr(df, disease=disease)
    if cfr_df.empty:
        continue
    high_cfr = cfr_df[cfr_df['flag'] == 'HIGH']
    nat_mean = cfr_df['national_mean_cfr'].iloc[0]
    print(f'  {disease:<20} National mean CFR: {nat_mean:.2f}%')
    if not high_cfr.empty:
        high_states = ', '.join(high_cfr['state'].tolist())
        print(f'    ⚠️  HIGH CFR states: {high_states}')

# Visualise CFR benchmark for cholera
cfr_cholera = benchmark_cfr(df, disease=Diseases.CHOLERA)
if not cfr_cholera.empty:
    fig = px.bar(
        cfr_cholera.sort_values('avg_cfr', ascending=False),
        x='state', y='avg_cfr', color='flag',
        color_discrete_map={'HIGH':'#E24B4A','NORMAL':'#1D9E75','LOW':'#85B7EB'},
        title='Cholera CFR by State vs National Mean',
        template='plotly_white',
    )
    fig.add_hline(
        y=cfr_cholera['national_mean_cfr'].iloc[0],
        line_dash='dash', line_color='grey',
        annotation_text='National mean'
    )
    fig.update_layout(height=380, xaxis_tickangle=-45)
    fig.show()

## 9. Summary

In [ ]:
from datetime import datetime
print('='*55)
print('  NOTEBOOK 04 — STATISTICAL ANALYSIS SUMMARY')
print('='*55)
print(f'  Timestamp : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()
print('  Key statistical findings (fill in after running):')
print('  • Mann-Kendall trend: (disease) shows (increasing/decreasing) trend')
print('  • Seasonality: Cholera is significantly higher in (Dry/Rainy) season')
print('  • Rainfall correlation: (positive/negative/none) with cholera')
print(f'  • Outbreak alerts detected: {len(all_alerts)} total')
print('  • State clustering: (description of cluster patterns)')
print()
print('  ➡️  Next: Notebook 05 — Geospatial Analysis')
print('='*55)